# Pilot: TabPFN-3.5 and LightGBM

Runs 50k, 100k, and 200k in order, projects the 700k TabPFN cost, and writes a PASS or FAIL safety report.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_REPO_URL='https://github.com/selimsidan/tfm-credit-benchmark.git'
GITHUB_BRANCH='main'
DRIVE_OUTPUT_DIR='/content/drive/MyDrive/tfm_credit_benchmark'
EXPERIMENT_NAME='lending_club_900k'
FORCE_REBUILD_SPLIT=False
FORCE_RERUN=False
RUN_TESTS=True
LIGHTGBM_TUNING_MINUTES=15
MAX_OPTUNA_TRIALS=60
COMPUTE_UNITS_PER_HOUR=5.4
MAX_SESSION_HOURS=20.0
# ==================================================================


In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import os, subprocess, sys
drive.mount('/content/drive')
repo=Path('/content/tfm-credit-benchmark')
if (repo/'.git').is_dir(): subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',GITHUB_BRANCH],check=True)
else: subprocess.run(['git','clone','--branch',GITHUB_BRANCH,'--single-branch',GITHUB_REPO_URL,str(repo)],check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-colab.txt'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
src_path=str(repo/'src')
if src_path not in sys.path: sys.path.insert(0,src_path)
import tfmbench
print('tfmbench package:',tfmbench.__file__)
try: tabpfn_token=userdata.get('TABPFN_TOKEN')
except Exception: tabpfn_token=None
if not tabpfn_token: raise RuntimeError('Missing TABPFN_TOKEN. Accept the TabPFN-3.5 license and grant this notebook access to the Colab secret.')
os.environ['TABPFN_TOKEN']=tabpfn_token; tabpfn_token=None
print('Repository commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())


In [ ]:
import json
from tfmbench.config import BenchmarkConfig
from tfmbench.resources import hardware_info, require_cuda
from tfmbench.split import verify_manifest
if RUN_TESTS: subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
hardware=require_cuda('The pilot')
print(json.dumps(hardware,indent=2))
config=BenchmarkConfig(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME); config.validate()
if FORCE_REBUILD_SPLIT:
    from tfmbench.data import prepare_data
    prepare_data(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,force_rebuild_split=True)
manifest=verify_manifest(config.shared_dir,config)
print('Verified manifest:',manifest['manifest_hash'])


In [ ]:
from tfmbench.pilot import run_pilot
report=run_pilot(DRIVE_OUTPUT_DIR,EXPERIMENT_NAME,force_rerun=FORCE_RERUN,tuning_minutes=LIGHTGBM_TUNING_MINUTES,max_trials=MAX_OPTUNA_TRIALS,compute_units_per_hour=COMPUTE_UNITS_PER_HOUR,max_session_hours=MAX_SESSION_HOURS)


In [ ]:
import pandas as pd
from tfmbench.evaluate import sibling_metrics_table
print('GO' if report['verdict']=='PASS' else 'NO-GO')
print((config.experiment_root/'pilot'/'pilot_report.md').read_text())
display(sibling_metrics_table(config.experiment_root))
